In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import re
print("Libraries imported successfully")

Libraries imported successfully


In [3]:
url = "https://books.toscrape.com/catalogue/category/books/travel_2/index.html"

response = requests.get(url)

print(response.status_code)
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

200

    Travel | 
     Books to Scrape - Sandbox




In [4]:
books = soup.select("article.product_pod")

print(len(books))

book = books[0]

print(book.prettify())

11
<article class="product_pod">
 <div class="image_container">
  <a href="../../../its-only-the-himalayas_981/index.html">
   <img alt="It's Only the Himalayas" class="thumbnail" src="../../../../media/cache/27/a5/27a53d0bb95bdd88288eaf66c9230d7e.jpg"/>
  </a>
 </div>
 <p class="star-rating Two">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="../../../its-only-the-himalayas_981/index.html" title="It's Only the Himalayas">
   It's Only the Himalayas
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£45.17
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [6]:
title = book.h3.a["title"]

print(title)

It's Only the Himalayas


In [7]:
price = book.select_one(".price_color").get_text(strip=True)

print(price)

Â£45.17


In [8]:
rating = book.select_one("p.star-rating")["class"][1]

print(rating)

Two


In [10]:
availability = book.select_one(".availability").get_text(" ", strip=True)

print(availability)
print("Title:", title)
print("Price:", price)
print("Rating:", rating)
print("Availability:", availability)

In stock
Title: It's Only the Himalayas
Price: Â£45.17
Rating: Two
Availability: In stock


In [11]:
#FUNCTION
def scrape_category(url, category_name):

    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.select("article.product_pod")

    results = []

    for book in books:

        # Title
        title = book.h3.a["title"]

        # Price
        price = book.select_one(".price_color").get_text(strip=True)

        # Rating
        rating = book.select_one("p.star-rating")["class"][1]

        # Availability
        availability = book.select_one(
            ".availability"
        ).get_text(" ", strip=True)

        results.append({
            "title": title,
            "price": price,
            "star_rating": rating,
            "availability": availability,
            "category": category_name
        })

    return results

In [13]:
travel_url = "https://books.toscrape.com/catalogue/category/books/travel_2/index.html"

travel_books = scrape_category(
    travel_url,
    "Travel"
)

print("Books scraped:", len(travel_books))
print(travel_books[:2])

Books scraped: 11
[{'title': "It's Only the Himalayas", 'price': 'Â£45.17', 'star_rating': 'Two', 'availability': 'In stock', 'category': 'Travel'}, {'title': 'Full Moon over Noahâ\x80\x99s Ark: An Odyssey to Mount Ararat and Beyond', 'price': 'Â£49.43', 'star_rating': 'Four', 'availability': 'In stock', 'category': 'Travel'}]


In [14]:
categories = {
    "Travel": "https://books.toscrape.com/catalogue/category/books/travel_2/index.html",

    "Mystery": "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html",

    "Historical Fiction": "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html",

    "Classics": "https://books.toscrape.com/catalogue/category/books/classics_6/index.html",

    "Science Fiction": "https://books.toscrape.com/catalogue/category/books/science-fiction_16/index.html"
}
all_books = []

for category_name, url in categories.items():

    books = scrape_category(url, category_name)

    print(category_name, ":", len(books), "books")

    all_books.extend(books)

print("\nTotal books scraped:", len(all_books))

Travel : 11 books
Mystery : 20 books
Historical Fiction : 20 books
Classics : 19 books
Science Fiction : 16 books

Total books scraped: 86


In [19]:
df = pd.DataFrame(all_books)

df.head()


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel


In [20]:
df.shape

(86, 5)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         86 non-null     object
 1   price         86 non-null     object
 2   star_rating   86 non-null     object
 3   availability  86 non-null     object
 4   category      86 non-null     object
dtypes: object(5)
memory usage: 3.5+ KB


In [21]:
df.isnull().sum()/86*100

,0
title,0.0
price,0.0
star_rating,0.0
availability,0.0
category,0.0


In [22]:
df.head(10)

,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel
5,A Summer In Europe,Â£44.34,Two,In stock,Travel
6,The Great Railway Bazaar,Â£30.54,One,In stock,Travel
7,A Year in Provence (Provence #1),Â£56.88,Four,In stock,Travel
8,The Road to Little Dribbling: Adventures of an...,Â£23.21,One,In stock,Travel
9,Neither Here nor There: Travels in Europe,Â£38.95,Three,In stock,Travel


In [26]:
#cleaning the price
df["price_gbp"] = (
    df["price"]
    .str.replace("Â£", "", regex=False).str.strip().astype(float)
)
df[["price", "price_gbp"]].head()

,price,price_gbp
0,Â£45.17,45.17
1,Â£49.43,49.43
2,Â£48.87,48.87
3,Â£36.94,36.94
4,Â£37.33,37.33


In [28]:
#rating changing into numbers
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)
df[["star_rating", "rating"]].head(10)

,star_rating,rating
0,Two,2
1,Four,4
2,Three,3
3,Two,2
4,Three,3
5,Two,2
6,One,1
7,Four,4
8,One,1
9,Three,3


In [31]:
#changing availability into boolean statement
df["in_stock"] = df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)
df[["availability", "in_stock"]].head()

,availability,in_stock
0,In stock,True
1,In stock,True
2,In stock,True
3,In stock,True
4,In stock,True


In [33]:
#checking missing values
print("Missing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nUnique ratings:")
print(df["rating"].unique())
print("\nInvalid price values:")
print(df[df["price_gbp"].isnull()])

Missing values:
title           0
price           0
star_rating     0
availability    0
category        0
price_gbp       0
rating          0
in_stock        0
dtype: int64

Data types:
title            object
price            object
star_rating      object
availability     object
category         object
price_gbp       float64
rating            int64
in_stock           bool
dtype: object

Unique ratings:
[2 4 3 1 5]

Invalid price values:
Empty DataFrame
Columns: [title, price, star_rating, availability, category, price_gbp, rating, in_stock]
Index: []


In [36]:
#converting into inr rupees
GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR
df[["price_gbp", "price_inr"]].head()

,price_gbp,price_inr
0,45.17,4765.435
1,49.43,5214.865
2,48.87,5155.785
3,36.94,3897.170
4,37.33,3938.315


In [39]:
clean_df = df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
].copy()
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      86 non-null     object 
 1   price_gbp  86 non-null     float64
 2   price_inr  86 non-null     float64
 3   rating     86 non-null     int64  
 4   in_stock   86 non-null     bool   
 5   category   86 non-null     object 
dtypes: bool(1), float64(2), int64(1), object(2)
memory usage: 3.6+ KB


In [40]:
clean_df.head()

,title,price_gbp,price_inr,rating,in_stock,category
0,It's Only the Himalayas,45.17,4765.435,2,True,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.865,4,True,Travel
2,See America: A Celebration of Our National Par...,48.87,5155.785,3,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170,2,True,Travel
4,Under the Tuscan Sun,37.33,3938.315,3,True,Travel


STORING IN DATABASE

In [43]:
conn = sqlite3.connect("books.db")

print("SQLite database created successfully")
create_categories_table = """
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
);
"""

conn.execute(create_categories_table)
conn.commit()

print("Categories table created")

SQLite database created successfully
Categories table created


In [42]:
create_books_table = """
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
);
"""

conn.execute(create_books_table)
conn.commit()

print("Books table created")

Books table created


In [45]:
unique_categories = clean_df["category"].unique()

print(unique_categories)
for category in unique_categories:

    conn.execute(
        """
        INSERT OR IGNORE INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )

conn.commit()
print("")

print("Categories inserted")

['Travel' 'Mystery' 'Historical Fiction' 'Classics' 'Science Fiction']

Categories inserted


In [59]:
categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

categories_df

,category_id,category_name
0,1,Travel
1,2,Mystery
2,3,Historical Fiction
3,4,Classics
4,5,Science Fiction


In [71]:
# Make category a string
clean_df["category"] = clean_df["category"].astype(str).str.strip()

# Split multiple category IDs
clean_df["category"] = clean_df["category"].str.split(",")

# Create separate row for each category ID
clean_df = clean_df.explode("category")

# Remove spaces
clean_df["category"] = clean_df["category"].str.strip()

# Convert category ID to integer
clean_df["category"] = pd.to_numeric(clean_df["category"], errors="coerce")
categories_df["category_id"] = pd.to_numeric(
    categories_df["category_id"],
    errors="coerce"
)
print(clean_df.columns.tolist())
clean_df = clean_df.drop(
    columns=[
        "category_id_x",
        "category_name_x",
        "category_id_y",
        "category_name_y"
    ],
    errors="ignore"
)
print(clean_df["category"].head(10).tolist())
print("Values in clean_df category:")
print(clean_df["category"].head(20).tolist())

print("\nValues in categories_df category_id:")
print(categories_df["category_id"].head(20).tolist())
clean_df = df.copy()
print(clean_df["category"].head(20).tolist())

['title', 'price', 'star_rating', 'availability', 'category', 'price_gbp', 'rating', 'in_stock', 'price_inr']
[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
Values in clean_df category:
[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]

Values in categories_df category_id:
[1, 2, 3, 4, 5]
['Travel', 'Travel', 'Travel', 'Travel', 'Travel', 'Travel', 'Travel', 'Travel', 'Travel', 'Travel', 'Travel', 'Mystery', 'Mystery', 'Mystery', 'Mystery', 'Mystery', 'Mystery', 'Mystery', 'Mystery', 'Mystery']


In [72]:
print(categories_df.head(20))

   category_id       category_name
0            1              Travel
1            2             Mystery
2            3  Historical Fiction
3            4            Classics
4            5     Science Fiction


In [73]:
print(df[["title", "category"]].head(10))
print(df["category"].unique())

                                               title category
0                            It's Only the Himalayas   Travel
1  Full Moon over Noahâs Ark: An Odyssey to Mou...   Travel
2  See America: A Celebration of Our National Par...   Travel
3  Vagabonding: An Uncommon Guide to the Art of L...   Travel
4                               Under the Tuscan Sun   Travel
5                                 A Summer In Europe   Travel
6                           The Great Railway Bazaar   Travel
7                   A Year in Provence (Provence #1)   Travel
8  The Road to Little Dribbling: Adventures of an...   Travel
9          Neither Here nor There: Travels in Europe   Travel
['Travel' 'Mystery' 'Historical Fiction' 'Classics' 'Science Fiction']


In [75]:
clean_df["category"] = clean_df["category"].astype(str).str.strip()
print(clean_df["category"].unique())

['Travel' 'Mystery' 'Historical Fiction' 'Classics' 'Science Fiction']


In [78]:
print("Clean DF categories:")
print(clean_df["category"].unique())

print("\nCategories DF names:")
print(categories_df["category_name"].unique())
clean_df = df.copy()
print(clean_df.columns.tolist())
clean_df = clean_df.merge(
    categories_df[["category_id", "category_name"]],
    left_on="category",
    right_on="category_name",
    how="left"
)

Clean DF categories:
['Travel' 'Mystery' 'Historical Fiction' 'Classics' 'Science Fiction']

Categories DF names:
['Travel' 'Mystery' 'Historical Fiction' 'Classics' 'Science Fiction']
['title', 'price', 'star_rating', 'availability', 'category', 'price_gbp', 'rating', 'in_stock', 'price_inr']


In [80]:
clean_df = clean_df.drop(columns=["category_name"])
print(clean_df.head())

                                               title    price star_rating  \
0                            It's Only the Himalayas  Â£45.17         Two   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...  Â£49.43        Four   
2  See America: A Celebration of Our National Par...  Â£48.87       Three   
3  Vagabonding: An Uncommon Guide to the Art of L...  Â£36.94         Two   
4                               Under the Tuscan Sun  Â£37.33       Three   

  availability category  price_gbp  rating  in_stock  price_inr  category_id  
0     In stock   Travel      45.17       2      True   4765.435            1  
1     In stock   Travel      49.43       4      True   5214.865            1  
2     In stock   Travel      48.87       3      True   5155.785            1  
3     In stock   Travel      36.94       2      True   3897.170            1  
4     In stock   Travel      37.33       3      True   3938.315            1  


In [82]:
books_to_insert = clean_df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id"
    ]
].copy()
books_to_insert["in_stock"] = (
    books_to_insert["in_stock"].astype(int)
)

In [84]:
books_to_insert.head()

,title,price_gbp,price_inr,rating,in_stock,category_id
0,It's Only the Himalayas,45.17,4765.435,2,1,1
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.865,4,1,1
2,See America: A Celebration of Our National Par...,48.87,5155.785,3,1,1
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170,2,1,1
4,Under the Tuscan Sun,37.33,3938.315,3,1,1


In [85]:
books_to_insert.to_sql(
    "books",
    conn,
    if_exists="append",
    index=False
)

print("Books inserted into database")

Books inserted into database


In [86]:
#verifying
result = pd.read_sql(
    """
    SELECT COUNT(*) AS total_books
    FROM books
    """,
    conn
)

result

,total_books
0,86


In [88]:
#sql query1
query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4;
"""

result1 = pd.read_sql(query1, conn)

result1

,title,price_gbp,rating
0,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,4
1,A Year in Provence (Provence #1),56.88,4
2,"1,000 Places to See Before You Die",26.08,5
3,Sharp Objects,47.82,4
4,The Past Never Ends,56.50,4
5,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4
6,A Time of Torment (Charlie Parker #14),48.35,5
7,Murder at the 42nd Street Library (Raymond Amb...,54.36,4
8,What Happened on Beale Street (Secrets of the ...,25.37,5
9,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5


In [89]:
query2 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC;
"""

result2 = pd.read_sql(query2, conn)

result2

,title,price_gbp,rating
0,Boar Island (Anna Pigeon #19),59.48,3
1,Candide,58.63,3
2,Animal Farm,57.22,3
3,A Year in Provence (Provence #1),56.88,4
4,The Past Never Ends,56.50,4
...,...,...,...
81,Playing with Fire,13.71,3
82,Hide Away (Eve Duncan #20),11.84,1
83,The Restaurant at the End of the Universe (Hit...,10.92,1
84,Tastes Like Fear (DI Marnie Rome #3),10.69,1


In [90]:
#distinct
query2 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC;
"""

result2 = pd.read_sql(query2, conn)

result2

,title,price_gbp,rating
0,Boar Island (Anna Pigeon #19),59.48,3
1,Candide,58.63,3
2,Animal Farm,57.22,3
3,A Year in Provence (Provence #1),56.88,4
4,The Past Never Ends,56.50,4
...,...,...,...
81,Playing with Fire,13.71,3
82,Hide Away (Eve Duncan #20),11.84,1
83,The Restaurant at the End of the Universe (Hit...,10.92,1
84,Tastes Like Fear (DI Marnie Rome #3),10.69,1


In [91]:
#join query
join_query = """
SELECT
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id;
"""

sql_join_result = pd.read_sql(
    join_query,
    conn
)

sql_join_result.head(10)

,title,price_gbp,price_inr,rating,category_name
0,It's Only the Himalayas,45.17,4765.435,2,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.865,4,Travel
2,See America: A Celebration of Our National Par...,48.87,5155.785,3,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170,2,Travel
4,Under the Tuscan Sun,37.33,3938.315,3,Travel
5,A Summer In Europe,44.34,4677.870,2,Travel
6,The Great Railway Bazaar,30.54,3221.970,1,Travel
7,A Year in Provence (Provence #1),56.88,6000.840,4,Travel
8,The Road to Little Dribbling: Adventures of an...,23.21,2448.655,1,Travel
9,Neither Here nor There: Travels in Europe,38.95,4109.225,3,Travel


In [92]:
import os

os.makedirs("query_outputs", exist_ok=True)

In [93]:
result1.to_csv(
    "query_outputs/query1_where.csv",
    index=False
)

result2.to_csv(
    "query_outputs/query2_order_by.csv",
    index=False
)

sql_join_result.to_csv(
    "query_outputs/query6_join.csv",
    index=False
)

In [94]:
clean_df.to_csv(
    "cleaned_books.csv",
    index=False
)

In [97]:
#final
print("=" * 50)
print("MODULE 1 FINAL VERIFICATION")
print("=" * 50)

total_books = pd.read_sql(
    "SELECT COUNT(*) AS count FROM books",
    conn
).iloc[0, 0]

total_categories = pd.read_sql(
    "SELECT COUNT(*) AS count FROM categories",
    conn
).iloc[0, 0]

print("Total books:", total_books)
print("Total categories:", total_categories)
print("Missing values:", clean_df.isnull().sum().sum())

print("\nDatabase tables:")

tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)

print(tables)

MODULE 1 FINAL VERIFICATION
Total books: 86
Total categories: 5
Missing values: 0

Database tables:
         name
0  categories
1       books


In [98]:
conn.close()

print("Database connection closed.")

Database connection closed.
